# Validate ResStock loads against EIA-861

This notebook loads ResStock building metadata, utility assignments, and annual load
curves for MD, then compares weighted residential totals to EIA-861
utility sales — first on customer counts, then on kWh after adjusting for
customer-count mismatch.

By default it reads metadata and annual loads from the **`_sb`** companion release
(`res_2024_amy2018_2_sb`); set `USE_SB_RELEASE = False` to read the same tables from
the raw NREL release (`res_2024_amy2018_2`) instead — e.g. to see how much
Switchbox's post-processing (MF non-HVAC scaling, non-HP approximation) moves the
comparison.

---

## How to use

1. Set `STATE` (lowercase 2-letter abbreviation) and `RESSTOCK_RELEASE` (the **raw**,
   non-`_sb` release name) in Parameters.
2. Set `USE_SB_RELEASE` (default `True`) to choose the `_sb` or raw release for
   metadata and annual load curves.
3. Optionally set `UPGRADE` (default `"00"`) and `EIA_YEAR` (default `2018`, to match
   ResStock AMY 2018).
4. Restart the kernel and run all cells.

---

## Data sources

| Input | S3 location |
|-------|-------------|
| Metadata | `s3://data.sb/nrel/resstock/<release>[_sb]/metadata/state=<ST>/upgrade=<uu>/metadata[-sb].parquet` |
| Utility assignment (always `_sb`) | `s3://data.sb/nrel/resstock/<release>_sb/metadata_utility/state=<ST>/utility_assignment.parquet` |
| Annual load curves | `s3://data.sb/nrel/resstock/<release>[_sb]/load_curve_annual/state=<ST>/upgrade=<uu>/<ST>_upgrade<uu>_metadata_and_annual_results.parquet` |
| EIA-861 utility stats | `s3://data.sb/eia/861/electric_utility_stats/year=<year>/state=<ST>/data.parquet` |

`USE_SB_RELEASE` controls **metadata** and **annual load curves** together:

- **`True` (default):** `metadata-sb.parquet` and the `_sb` annual file. Both reflect
  Switchbox's post-processing adjustments (e.g. MF non-HVAC scaling, non-HP
  approximation); the `_sb` annual values match the sum of the `_sb` monthly/hourly
  load curves.
- **`False`:** raw `metadata.parquet` and the raw NREL annual file — unmodified NREL
  simulation output, useful for isolating how much the `_sb` adjustments move the
  ResStock-vs-EIA comparison.

**Utility assignment never toggles.** `utility_assignment.parquet` is a Switchbox-only
artifact generated only under the `_sb` release — it has no raw-release counterpart
(see `context/code/data/resstock_sb_release_pipeline_main_py.md`) — so it's always read
from `<release>_sb`, regardless of `USE_SB_RELEASE`.

Either way, annual electricity is read directly from
`out.electricity.total.energy_consumption.kwh`; sample `weight` comes from whichever
metadata file `USE_SB_RELEASE` selects.

---

## Notebook sections

| Section | What it covers |
|---------|---------------|
| **1 — Load data** | Read metadata, utility assignment, and annual loads; keep the `bldg_id` intersection |
| **2 — Sum by utility** | Weighted electricity totals by `sb.electric_utility` |
| **3 — Compare to EIA** | Customer-count ratios; then kWh comparison after scaling ResStock to EIA customer counts |
| **4 — Assumptions** | *(todo)* Document limitations and interpretation guidance |

## Parameters

Change `STATE`, `RESSTOCK_RELEASE`, and `USE_SB_RELEASE` here. Everything else derives from these values.

In [121]:
from __future__ import annotations

from typing import cast

import polars as pl
from IPython.display import display

In [122]:
# ── Change these to validate a different state or release ─────────────────────
STATE = "md"  # lowercase state abbreviation, e.g. "ri", "ny", "ma"
RESSTOCK_RELEASE = "res_2024_amy2018_2"  # base (raw, non-_sb) release name
UPGRADE = "00"  # zero-padded ResStock upgrade id
EIA_YEAR = 2018  # EIA-861 report year (2018 aligns with ResStock AMY 2018)
# True: read metadata + annual loads from the _sb release ("metadata-sb.parquet", _sb
# annual). False: read them from the raw NREL release ("metadata.parquet", raw annual).
USE_SB_RELEASE = True
# ──────────────────────────────────────────────────────────────────────────────

STATE_UPPER = STATE.upper()
RESSTOCK_RELEASE_RAW = RESSTOCK_RELEASE.removesuffix("_sb")
RESSTOCK_RELEASE_SB = f"{RESSTOCK_RELEASE_RAW}_sb"

# Metadata and annual loads toggle between raw and _sb per USE_SB_RELEASE.
# utility_assignment.parquet is a Switchbox-only artifact that is never written under
# the raw release (see context/code/data/resstock_sb_release_pipeline_main_py.md), so
# it always comes from _sb regardless of the toggle.
RESSTOCK_RELEASE_ACTIVE = RESSTOCK_RELEASE_SB if USE_SB_RELEASE else RESSTOCK_RELEASE_RAW
METADATA_FILENAME = "metadata-sb.parquet" if USE_SB_RELEASE else "metadata.parquet"
RELEASE_SOURCE_LABEL = "_sb (post-processed)" if USE_SB_RELEASE else "raw NREL (unmodified)"

S3_RESSTOCK = "s3://data.sb/nrel/resstock"
S3_EIA861 = "s3://data.sb/eia/861/electric_utility_stats"

BLDG_ID = "bldg_id"
UTILITY_COL = "sb.electric_utility"
WEIGHT_COL = "weight"
# Annual electricity total (".kwh" suffix on the annual file; same column name in raw and _sb)
ANNUAL_ELEC_COL = "out.electricity.total.energy_consumption.kwh"

PATH_METADATA = (
    f"{S3_RESSTOCK}/{RESSTOCK_RELEASE_ACTIVE}/metadata/state={STATE_UPPER}/upgrade={UPGRADE}/{METADATA_FILENAME}"
)
# Always _sb: utility_assignment.parquet has no raw-release counterpart.
PATH_UTILITY_ASSIGNMENT = (
    f"{S3_RESSTOCK}/{RESSTOCK_RELEASE_SB}/metadata_utility/state={STATE_UPPER}/utility_assignment.parquet"
)
PATH_ANNUAL = (
    f"{S3_RESSTOCK}/{RESSTOCK_RELEASE_ACTIVE}/load_curve_annual/"
    f"state={STATE_UPPER}/upgrade={UPGRADE}/"
    f"{STATE_UPPER}_upgrade{UPGRADE}_metadata_and_annual_results.parquet"
)
PATH_EIA861 = f"{S3_EIA861}/year={EIA_YEAR}/state={STATE_UPPER}/data.parquet"

print(f"State:              {STATE_UPPER}")
print(f"Release source:     {RESSTOCK_RELEASE_ACTIVE} ({RELEASE_SOURCE_LABEL})")
print(f"Utility assignment: {RESSTOCK_RELEASE_SB} (always _sb; Switchbox-only, no raw equivalent)")
print(f"Upgrade:            {UPGRADE}")
print(f"EIA-861 year:       {EIA_YEAR}")
print()
print(f"Metadata:           {PATH_METADATA}")
print(f"Utility assignment: {PATH_UTILITY_ASSIGNMENT}")
print(f"Annual load curves: {PATH_ANNUAL}")
print(f"EIA-861 stats:      {PATH_EIA861}")

State:              MD
Release source:     res_2024_amy2018_2_sb (_sb (post-processed))
Utility assignment: res_2024_amy2018_2_sb (always _sb; Switchbox-only, no raw equivalent)
Upgrade:            00
EIA-861 year:       2018

Metadata:           s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/metadata/state=MD/upgrade=00/metadata-sb.parquet
Utility assignment: s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/metadata_utility/state=MD/utility_assignment.parquet
Annual load curves: s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/load_curve_annual/state=MD/upgrade=00/MD_upgrade00_metadata_and_annual_results.parquet
EIA-861 stats:      s3://data.sb/eia/861/electric_utility_stats/year=2018/state=MD/data.parquet


## Section 1: Load data

We read three ResStock tables from S3. Metadata and annual load curves come from
either the raw NREL release or the `_sb` release, per `USE_SB_RELEASE`; utility
assignment always comes from `_sb` (see Parameters):

1. **Metadata** (`metadata-sb.parquet` or raw `metadata.parquet`) — building
   attributes and sample `weight` (one row per `bldg_id` for this upgrade).
2. **`utility_assignment.parquet`** — `sb.electric_utility` / `sb.gas_utility` per building.
3. **`load_curve_annual`** — a single parquet with one row per `bldg_id` carrying
   `out.electricity.total.energy_consumption.kwh` (the annual electricity total). We
   read this directly as `annual_kwh` — no monthly aggregation needed.

After loading, we print shape, schema, and a small sample so the reader can confirm
the expected columns are present before aggregation.

> **Note:** The `_sb` versions of metadata and annual load curves reflect Switchbox's
> post-processing adjustments and (for annual) match the sum of the `_sb`
> monthly/hourly load curves; the raw NREL versions do not. Reading the single annual
> parquet is far faster than fetching thousands of monthly files, whichever source
> `USE_SB_RELEASE` selects.

In [123]:
def load_parquet(path: str) -> pl.DataFrame:
    """Collect a parquet file (or Hive directory) from S3 or local disk."""
    return cast(pl.DataFrame, pl.scan_parquet(path).collect())


def preview(name: str, df: pl.DataFrame, key_cols: list[str] | None = None) -> None:
    """Print shape, optional key-column check, schema head, and a few rows."""
    print(f"=== {name} ===")
    print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} cols")
    if key_cols:
        missing = [c for c in key_cols if c not in df.columns]
        if missing:
            raise ValueError(f"{name} is missing required columns: {missing}")
        print(f"required columns present: {key_cols}")
    print("schema (first 25):")
    for col, dtype in list(df.schema.items())[:25]:
        print(f"  {col}: {dtype}")
    if len(df.schema) > 25:
        print(f"  … ({len(df.schema) - 25} more columns)")
    display(df.head(5))
    print()

In [124]:
print(f"Loading metadata from:\n  {PATH_METADATA}\n")
metadata = load_parquet(PATH_METADATA)
preview(f"metadata ({RELEASE_SOURCE_LABEL})", metadata, key_cols=[BLDG_ID, WEIGHT_COL])

Loading metadata from:
  s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/metadata/state=MD/upgrade=00/metadata-sb.parquet

=== metadata (_sb (post-processed)) ===
shape: 9,996 rows x 188 cols
required columns present: ['bldg_id', 'weight']
schema (first 25):
  upgrade: Int64
  weight: Float64
  in.sqft: Int64
  in.representative_income: Float64
  in.ahs_region: String
  in.aiannh_area: String
  in.area_median_income: String
  in.ashrae_iecc_climate_zone_2004: String
  in.ashrae_iecc_climate_zone_2004_2_a_split: String
  in.bathroom_spot_vent_hour: String
  in.battery: String
  in.bedrooms: String
  in.building_america_climate_zone: String
  in.cec_climate_zone: String
  in.ceiling_fan: String
  in.census_division: String
  in.census_division_recs: String
  in.census_region: String
  in.city: String
  in.clothes_dryer: String
  in.clothes_dryer_usage_level: String
  in.clothes_washer: String
  in.clothes_washer_presence: String
  in.clothes_washer_usage_level: String
  in.cooking_range

upgrade,weight,in.sqft,in.representative_income,in.ahs_region,in.aiannh_area,in.area_median_income,in.ashrae_iecc_climate_zone_2004,in.ashrae_iecc_climate_zone_2004_2_a_split,in.bathroom_spot_vent_hour,in.battery,in.bedrooms,in.building_america_climate_zone,in.cec_climate_zone,in.ceiling_fan,in.census_division,in.census_division_recs,in.census_region,in.city,in.clothes_dryer,in.clothes_dryer_usage_level,in.clothes_washer,in.clothes_washer_presence,in.clothes_washer_usage_level,in.cooking_range,in.cooking_range_usage_level,in.cooling_setpoint,in.cooling_setpoint_has_offset,in.cooling_setpoint_offset_magnitude,in.cooling_setpoint_offset_period,in.corridor,in.county,in.county_and_puma,in.county_name,in.dehumidifier,in.dishwasher,in.dishwasher_usage_level,…,in.solar_hot_water,in.state,in.tenure,in.units_represented,in.usage_level,in.utility_bill_electricity_fixed_charges,in.utility_bill_electricity_marginal_rates,in.utility_bill_fuel_oil_fixed_charges,in.utility_bill_fuel_oil_marginal_rates,in.utility_bill_natural_gas_fixed_charges,in.utility_bill_natural_gas_marginal_rates,in.utility_bill_propane_fixed_charges,in.utility_bill_propane_marginal_rates,in.utility_bill_scenario_names,in.utility_bill_simple_filepaths,in.vacancy_status,in.vintage,in.vintage_acs,in.water_heater_efficiency,in.water_heater_fuel,in.water_heater_in_unit,in.water_heater_location,in.weather_file_city,in.weather_file_latitude,in.weather_file_longitude,in.window_areas,in.windows,bldg_id,postprocess_group.has_hp,postprocess_group.heating_type,postprocess_group.heating_type_v2,heats_with_electricity,heats_with_natgas,heats_with_oil,heats_with_propane,has_natgas_connection,mf_non_hvac_electricity_adjusted
i64,f64,i64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,i64,str,i64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,i64,bool,str,str,bool,bool,bool,bool,bool,bool
0,252.301639,1698,108692.0,"""Non-CBSA South Atlantic""","""No""","""120-150%""","""4A""","""4A""","""Hour20""","""None""","""3""","""Mixed-Humid""","""None""","""Standard Efficiency""","""South Atlantic""","""South Atlantic""","""South""","""In another census Place""","""Gas""","""100% Usage""","""EnergyStar""","""Yes""","""100% Usage""","""Electric Resistance""","""100% Usage""","""70F""","""Yes""","""9F""","""Night Setup +4h""","""Not Applicable""","""G2400050""","""G2400050, G24000505""","""Baltimore County""","""None""","""318 Rated kWh""","""100% Usage""",…,"""None""","""MD""","""Owner""",1,"""Medium""",10,0.130352,"""0""","""3.219615385""","""11.25""","""1.217491908""","""0""","""3.224769231""","""Utility Rates - Fixed + Variab…","""data/simple_rates/State.tsv""","""Occupied""","""2000s""","""2000-09""","""Natural Gas Standard""","""Natural Gas""","""Yes""","""Living Space""","""Baltimore Washingto""",39.17,-76.68,"""F18 B18 L18 R18""","""Double, Clear, Metal, Air""",434236,false,"""fossil_fuel""","""natgas""",false,true,false,false,true,false
0,252.301639,2179,89096.0,"""Non-CBSA South Atlantic""","""No""","""100-120%""","""4A""","""4A""","""Hour20""","""None""","""3""","""Mixed-Humid""","""None""","""Standard Efficiency""","""South Atlantic""","""South Atlantic""","""South""","""Not in a census Place""","""Electric""","""100% Usage""","""EnergyStar""","""Yes""","""100% Usage""","""Electric Induction""","""100% Usage""","""72F""","""No""","""0F""","""None""","""Not Applicable""","""G2400030""","""G2400030, G24001203""","""Anne Arundel County""","""None""","""318 Rated kWh""","""100% Usage""",…,"""None""","""MD""","""Owner""",1,"""Medium""",10,0.130352,"""0""","""3.219615385""","""11.25""","""1.217491908""","""0""","""3.224769231""","""Utility Rates - Fixed + Variab…","""data/simple_rates/State.tsv""","""Occupied""","""1970s""","""1960-79""","""Natural Gas Standard""","""Natural Gas""","""Yes""","""Heated Basement""","""Baltimore Washingto""",39.17,-76.68,""

In [125]:
print(f"Loading utility assignment from:\n  {PATH_UTILITY_ASSIGNMENT}\n")
utility_assignment = load_parquet(PATH_UTILITY_ASSIGNMENT)
preview(
    "utility_assignment",
    utility_assignment,
    key_cols=[BLDG_ID, UTILITY_COL],
)

Loading utility assignment from:
  s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/metadata_utility/state=MD/utility_assignment.parquet

=== utility_assignment ===
shape: 9,996 rows x 3 cols
required columns present: ['bldg_id', 'sb.electric_utility']
schema (first 25):
  bldg_id: Int64
  sb.electric_utility: String
  sb.gas_utility: String


bldg_id,sb.electric_utility,sb.gas_utility
i64,str,str
434236,"""bge""","""bge"""
436422,"""bge""","""bge"""
439752,"""choptank""",null
443384,"""smeco""","""washington_gas"""
443437,"""dpl""",null


In [126]:
print(f"Loading annual load curves ({RELEASE_SOURCE_LABEL}):\n  {PATH_ANNUAL}\n")
annual = load_parquet(PATH_ANNUAL).select(
    BLDG_ID,
    pl.col(ANNUAL_ELEC_COL).alias("annual_kwh"),
)
preview(f"load_curve_annual ({RELEASE_SOURCE_LABEL})", annual, key_cols=[BLDG_ID, "annual_kwh"])

Loading annual load curves (_sb (post-processed)):
  s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/load_curve_annual/state=MD/upgrade=00/MD_upgrade00_metadata_and_annual_results.parquet

=== load_curve_annual (_sb (post-processed)) ===
shape: 9,995 rows x 2 cols
required columns present: ['bldg_id', 'annual_kwh']
schema (first 25):
  bldg_id: Int64
  annual_kwh: Float64


bldg_id,annual_kwh
i64,f64
100081,18829.255
100076,24075.954
100135,12097.12
100225,14836.117
100337,12213.863


### Align on shared buildings

Metadata, utility assignment, and annual loads should each have one row per building
for this state/upgrade. Small mismatches can occur when a building is present in one
table but not another (e.g. off-by-one row-count differences between the annual file
and metadata).

For the rest of this notebook we keep only the **intersection** of `bldg_id`s across
all three tables, and report any IDs that were dropped.

In [127]:
n_meta = metadata.height
n_ua = utility_assignment.height
n_annual = annual.height
n_meta_ids = metadata[BLDG_ID].n_unique()
n_ua_ids = utility_assignment[BLDG_ID].n_unique()
n_annual_ids = annual[BLDG_ID].n_unique()

print("Before intersection:")
print(f"  metadata:           {n_meta:,} rows, {n_meta_ids:,} unique {BLDG_ID}")
print(f"  utility_assignment: {n_ua:,} rows, {n_ua_ids:,} unique {BLDG_ID}")
print(f"  annual ({RELEASE_SOURCE_LABEL}):       {n_annual:,} rows, {n_annual_ids:,} unique {BLDG_ID}")

if n_meta != n_meta_ids:
    print("WARNING: metadata has duplicate bldg_id values")
if n_ua != n_ua_ids:
    print("WARNING: utility_assignment has duplicate bldg_id values")
if n_annual != n_annual_ids:
    print("WARNING: annual ({RELEASE_SOURCE_LABEL}) has duplicate bldg_id values")

meta_ids = set(metadata[BLDG_ID].to_list())
ua_ids = set(utility_assignment[BLDG_ID].to_list())
annual_ids = set(annual[BLDG_ID].to_list())
shared_ids = meta_ids & ua_ids & annual_ids

dropped_meta = sorted(meta_ids - shared_ids)
dropped_ua = sorted(ua_ids - shared_ids)
dropped_annual = sorted(annual_ids - shared_ids)

print(f"\nShared bldg_ids (metadata ∩ utility_assignment ∩ annual): {len(shared_ids):,}")
print(f"  dropped from metadata only:           {len(dropped_meta):,} {dropped_meta[:10]}")
print(f"  dropped from utility_assignment only: {len(dropped_ua):,} {dropped_ua[:10]}")
print(f"  dropped from annual only:             {len(dropped_annual):,} {dropped_annual[:10]}")

shared_ids_list = list(shared_ids)
metadata = metadata.filter(pl.col(BLDG_ID).is_in(shared_ids_list))
utility_assignment = utility_assignment.filter(pl.col(BLDG_ID).is_in(shared_ids_list))
annual = annual.filter(pl.col(BLDG_ID).is_in(shared_ids_list))

print("\nAfter intersection:")
print(f"  metadata:           {metadata.height:,}")
print(f"  utility_assignment: {utility_assignment.height:,}")
print(f"  annual ({RELEASE_SOURCE_LABEL}):       {annual.height:,}")
assert metadata.height == utility_assignment.height == annual.height
assert metadata[BLDG_ID].n_unique() == metadata.height

Before intersection:
  metadata:           9,996 rows, 9,996 unique bldg_id
  utility_assignment: 9,996 rows, 9,996 unique bldg_id
  annual (_sb (post-processed)):       9,995 rows, 9,995 unique bldg_id

Shared bldg_ids (metadata ∩ utility_assignment ∩ annual): 9,995
  dropped from metadata only:           1 [7506]
  dropped from utility_assignment only: 1 [7506]
  dropped from annual only:             0 []

After intersection:
  metadata:           9,995
  utility_assignment: 9,995
  annual (_sb (post-processed)):       9,995


## Section 2: Sum electricity by utility

Join the aligned annual loads to utility assignment and metadata `weight` on
`bldg_id`, then for each `sb.electric_utility` sum **weighted** annual electricity:

$$\text{resstock\_total\_kwh} = \sum_i (\text{annual\_kwh}_i \times \text{weight}_i)$$

ResStock `weight` is the sample expansion factor from the metadata file
selected by `USE_SB_RELEASE` (each building
represents roughly 252 dwellings). Customer counts are likewise
$\sum_i \text{weight}_i$.

Buildings with a null electric utility assignment are reported separately and
excluded from the per-utility totals.

In [128]:
annual_with_utility = (
    annual.select(BLDG_ID, "annual_kwh")
    .join(
        metadata.select(BLDG_ID, WEIGHT_COL),
        on=BLDG_ID,
        how="inner",
        validate="1:1",
    )
    .join(
        utility_assignment.select(BLDG_ID, UTILITY_COL),
        on=BLDG_ID,
        how="inner",
        validate="1:1",
    )
    .with_columns((pl.col("annual_kwh") * pl.col(WEIGHT_COL)).alias("weighted_kwh"))
)

n_null_utility = annual_with_utility.filter(pl.col(UTILITY_COL).is_null()).height
if n_null_utility:
    print(f"WARNING: {n_null_utility:,} buildings have null {UTILITY_COL}; excluded from totals")

resstock_by_utility = (
    annual_with_utility.filter(pl.col(UTILITY_COL).is_not_null())
    .group_by(UTILITY_COL)
    .agg(
        pl.len().alias("buildings"),
        pl.col(WEIGHT_COL).sum().alias("resstock_customers"),
        pl.col("weighted_kwh").sum().alias("resstock_total_kwh"),
    )
    .sort("resstock_total_kwh", descending=True)
    .rename({UTILITY_COL: "utility_code"})
)

resstock_by_utility_with_total = pl.concat(
    [
        resstock_by_utility,
        pl.DataFrame(
            {
                "utility_code": ["**TOTAL**"],
                "buildings": [resstock_by_utility["buildings"].sum()],
                "resstock_customers": [resstock_by_utility["resstock_customers"].sum()],
                "resstock_total_kwh": [resstock_by_utility["resstock_total_kwh"].sum()],
            },
            schema=resstock_by_utility.schema,
        ),
    ]
)

print(f"ResStock weighted electricity by {UTILITY_COL} ({STATE_UPPER}, upgrade {UPGRADE}, {RELEASE_SOURCE_LABEL}):")
display(resstock_by_utility_with_total)

ResStock weighted electricity by sb.electric_utility (MD, upgrade 00, _sb (post-processed)):


utility_code,buildings,resstock_customers,resstock_total_kwh
str,u32,f64,f64
"""bge""",4897,1.2355e6,1.5982e10
"""pepco""",2344,591395.041184,7.3058e9
"""poted""",1076,271476.563274,4.0710e9
"""smeco""",630,158950.0324,2.7047e9
"""dpl""",546,137756.694747,1.9062e9
"""choptank""",475,119843.278397,1.5681e9
"""hagerstown_muni""",25,6307.540968,1.1014e8
"""berlin_muni""",1,252.301639,3.9837e6
"""easton_muni""",1,252.301639,3.0390e6


## Section 3: Compare to EIA-861

We compare ResStock to **EIA-861 residential sales** in two steps:

1. **Customer counts** — `resstock_customers` (sum of sample weights) vs
   `eia_residential_customers`, and their ratio.
2. **Electricity (kWh)** — scale ResStock total kWh by the inverse customer ratio
   so the kWh comparison is not driven by customer-count mismatch, then report
   ratios and % differences.

The join key is a `utility_code` (short std_name, e.g. `bge`) that both
sides share: ResStock's `sb.electric_utility` on one side, and EIA-861's
`utility_code` column on the other. The EIA-861 parquet's `utility_code` is
derived at *fetch time* from `rate-design-platform`'s
[`utils/utility_codes.py`](https://github.com/switchbox-data/rate-design-platform/blob/main/utils/utility_codes.py)
crosswalk (EIA `utility_id_eia` → std_name) — if a utility was added to that
crosswalk *after* the EIA-861 parquet for this state was last fetched,
`utility_code` will be `null` for it even though `utility_id_eia` is present.

To make this notebook robust to a stale/partial `utility_code` column, we build
our own `utility_id_eia → utility_code` map below (mirroring the current
`utils/utility_codes.py`) and use it to fill in any missing values before
aggregating. **This map is state-specific — update it when switching `STATE`.**

In [129]:
print(f"Loading EIA-861 residential sales from:\n  {PATH_EIA861}\n")
eia_raw = load_parquet(PATH_EIA861)
preview(
    "eia861",
    eia_raw,
    key_cols=["utility_id_eia", "utility_code", "residential_sales_mwh", "residential_customers"],
)

n_null_code = eia_raw.filter(pl.col("utility_code").is_null()).height
print(f"Rows with null utility_code: {n_null_code:,} of {eia_raw.height:,}")

Loading EIA-861 residential sales from:
  s3://data.sb/eia/861/electric_utility_stats/year=2018/state=MD/data.parquet

=== eia861 ===
shape: 92 rows x 25 cols
required columns present: ['utility_id_eia', 'utility_code', 'residential_sales_mwh', 'residential_customers']
schema (first 25):
  year: Int32
  state: String
  utility_id_eia: Int64
  utility_code: String
  utility_name: String
  business_model: Categorical
  entity_type: String
  report_date: Date
  total_sales_mwh: Float32
  total_sales_revenue: Float32
  commercial_sales_mwh: Float32
  commercial_sales_revenue: Float32
  commercial_customers: Float32
  industrial_sales_mwh: Float32
  industrial_sales_revenue: Float32
  industrial_customers: Float32
  other_sales_mwh: Float32
  other_sales_revenue: Float32
  other_customers: Float32
  residential_sales_mwh: Float32
  residential_sales_revenue: Float32
  residential_customers: Float32
  transportation_sales_mwh: Float32
  transportation_sales_revenue: Float32
  transportation_

year,state,utility_id_eia,utility_code,utility_name,business_model,entity_type,report_date,total_sales_mwh,total_sales_revenue,commercial_sales_mwh,commercial_sales_revenue,commercial_customers,industrial_sales_mwh,industrial_sales_revenue,industrial_customers,other_sales_mwh,other_sales_revenue,other_customers,residential_sales_mwh,residential_sales_revenue,residential_customers,transportation_sales_mwh,transportation_sales_revenue,transportation_customers
i32,str,i64,str,str,cat,str,date,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
2018,"""MD""",17710,null,"""Spark Energy, LP""","""retail""","""Retail Power Marketer""",2018-01-01,46166.0,5.5436e6,11943.0,1.068e6,114.0,0.0,0.0,0.0,0.0,0.0,0.0,34223.0,4.4756e6,3167.0,0.0,0.0,0.0
2018,"""MD""",59139,null,"""SunEdison LLC""","""retail""","""Behind the Meter""",2018-01-01,13.0,1400.0,13.0,1400.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2018,"""MD""",84,null,"""A & N Electric Coop""","""retail""","""Cooperative""",2018-01-01,2901.0,353400.0,589.0,73500.0,43.0,0.0,0.0,0.0,0.0,0.0,0.0,2312.0,279900.0,271.0,0.0,0.0,0.0
2018,"""MD""",58162,null,"""Viridian Energy PA LLC""","""retail""","""Retail Power Marketer""",2018-01-01,66027.0,7.6912e6,14574.0,1.5474e6,592.0,0.0,0.0,0.0,0.0,0.0,0.0,51453.0,6.1438e6,4238.0,0.0,0.0,0.0
2018,"""MD""",61823,null,"""Greenskies Renewable Energy, L…","""retail""","""Behind the Meter""",2018-01-01,3222.0,346600.0,3222.0,346600.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Rows with null utility_code: 92 of 92


### Fill in missing `utility_code` values

The map below is `state`-specific: it lists every `utility_id_eia` we expect for
`STATE` and the `sb.electric_utility` std_name it corresponds to, taken from
`rate-design-platform`'s `utils/utility_codes.py`. We use it only to fill in rows
where `utility_code` is currently `null` — any row that already has a non-null
`utility_code` is left as-is.

**When running this notebook for a different state**, replace this dict with the
`(state=STATE)` entries from `utils/utility_codes.py` (`std_name` →
`eia_utility_ids`), or skip this cell if that state's `utility_code` column is
already fully populated.

In [130]:
# utility_id_eia -> sb.electric_utility std_name, for STATE = "md".
# Source: rate-design-platform/utils/utility_codes.py (UTILITIES list, state="MD").
EIA_UTILITY_ID_TO_STD_NAME = {
    1167: "bge",
    15270: "pepco",
    15263: "poted",
    5027: "dpl",
    17637: "smeco",
    3503: "choptank",
    40167: "somerset_rec",
    1615: "berlin_muni",
    7908: "hagerstown_muni",
    5625: "easton_muni",
}

eia_filled = eia_raw.with_columns(
    pl.coalesce(
        pl.col("utility_code"),
        pl.col("utility_id_eia").replace_strict(EIA_UTILITY_ID_TO_STD_NAME, default=None),
    ).alias("utility_code")
)

n_filled = (
    eia_filled.filter(pl.col("utility_code").is_not_null()).height
    - eia_raw.filter(pl.col("utility_code").is_not_null()).height
)
print(f"Filled in utility_code for {n_filled:,} rows using EIA_UTILITY_ID_TO_STD_NAME.")

n_still_null_with_sales = eia_filled.filter(
    pl.col("utility_code").is_null() & (pl.col("residential_sales_mwh") > 0)
).height
if n_still_null_with_sales:
    print(
        f"WARNING: {n_still_null_with_sales:,} utilities with residential sales still have "
        f"no utility_code (not in EIA_UTILITY_ID_TO_STD_NAME) and will be excluded below."
    )

eia_by_utility = (
    eia_filled.filter(pl.col("utility_code").is_not_null())
    .group_by("utility_code")
    .agg(
        (pl.col("residential_sales_mwh").sum() * 1000).alias("eia_residential_kwh"),
        pl.col("residential_customers").sum().alias("eia_residential_customers"),
    )
)

print(f"\nEIA-861 residential sales by utility_code ({STATE_UPPER}, {EIA_YEAR}):")
display(eia_by_utility.sort("eia_residential_kwh", descending=True))

Filled in utility_code for 8 rows using EIA_UTILITY_ID_TO_STD_NAME.

EIA-861 residential sales by utility_code (MD, 2018):


utility_code,eia_residential_kwh,eia_residential_customers
str,f32,f32
"""bge""",1.2948e10,1.164647e6
"""pepco""",5.8435e9,525580.0
"""poted""",3.3581e9,236072.0
"""smeco""",2.2768e9,150167.0
"""dpl""",2.2193e9,178457.0
"""choptank""",7.18323968e8,48571.0
"""hagerstown_muni""",1.66743008e8,14863.0
"""easton_muni""",1.14926e8,8355.0


### Customer-count comparison

First we join ResStock and EIA on `utility_code` and look **only** at customer
counts. ResStock customers are $\sum_i \text{weight}_i$; EIA customers are
`residential_customers` from EIA-861.

$$\text{customers\_ratio} = \frac{\text{resstock\_customers}}{\text{eia\_residential\_customers}}$$

A ratio near 1 means the sample expansion matches EIA's reported customer base
for that utility. Ratios far from 1 (or `inf` when EIA reports 0 customers)
mean later kWh comparisons must be customer-normalized.

In [131]:
joined = resstock_by_utility.join(eia_by_utility, on="utility_code", how="inner")

n_dropped = resstock_by_utility.height - joined.height
if n_dropped:
    dropped_codes = sorted(set(resstock_by_utility["utility_code"]) - set(joined["utility_code"]))
    print(
        f"WARNING: {n_dropped} ResStock utilities had no EIA-861 match and were "
        f"dropped from the comparison: {dropped_codes}"
    )

customers_comparison = joined.select(
    "utility_code",
    "buildings",
    "resstock_customers",
    "eia_residential_customers",
    (pl.col("resstock_customers") / pl.col("eia_residential_customers")).alias("customers_ratio"),
    (
        (pl.col("resstock_customers") - pl.col("eia_residential_customers")) / pl.col("eia_residential_customers") * 100
    ).alias("customers_pct_diff"),
).sort("resstock_customers", descending=True)

print(f"ResStock vs EIA-861 customer counts by utility ({STATE_UPPER}, {EIA_YEAR}):")
display(customers_comparison)

ResStock vs EIA-861 customer counts by utility (MD, 2018):


utility_code,buildings,resstock_customers,eia_residential_customers,customers_ratio,customers_pct_diff
str,u32,f64,f32,f64,f64
"""bge""",4897,1.2355e6,1.164647e6,1.060855,6.08546
"""pepco""",2344,591395.041184,525580.0,1.125224,12.522364
"""poted""",1076,271476.563274,236072.0,1.149974,14.997358
"""smeco""",630,158950.0324,150167.0,1.058488,5.848843
"""dpl""",546,137756.694747,178457.0,0.771932,-22.806786
"""choptank""",475,119843.278397,48571.0,2.467383,146.738339
"""hagerstown_muni""",25,6307.540968,14863.0,0.424379,-57.562128
"""easton_muni""",1,252.301639,8355.0,0.030198,-96.980232


### kWh comparison (customer-count normalized)

To isolate differences in **per-customer electricity use**, we scale ResStock
weighted total kWh to EIA's customer count before comparing:

$$
\text{resstock\_total\_kwh\_normalized} =
\text{resstock\_total\_kwh} \times
\frac{\text{eia\_residential\_customers}}{\text{resstock\_customers}}
= \frac{\text{resstock\_total\_kwh}}{\text{customers\_ratio}}
$$

Then:

- `kwh_ratio` = normalized ResStock kWh / EIA residential kWh  
- `kwh_pct_diff` = (normalized ResStock − EIA) / EIA × 100

Utilities with zero EIA customers (or zero ResStock customers) cannot be
normalized this way and are excluded from the kWh table below.

In [132]:
kwh_base = joined.filter(
    (pl.col("eia_residential_customers") > 0) & (pl.col("resstock_customers") > 0) & (pl.col("eia_residential_kwh") > 0)
)

n_excluded_kwh = joined.height - kwh_base.height
if n_excluded_kwh:
    excluded_codes = sorted(set(joined["utility_code"]) - set(kwh_base["utility_code"]))
    print(
        f"Excluded {n_excluded_kwh} utilities from kWh comparison "
        f"(zero EIA/ResStock customers or zero EIA kWh): {excluded_codes}"
    )

kwh_comparison = (
    kwh_base.with_columns(
        (pl.col("resstock_total_kwh") * pl.col("eia_residential_customers") / pl.col("resstock_customers")).alias(
            "resstock_total_kwh_normalized"
        ),
        (pl.col("resstock_customers") / pl.col("eia_residential_customers")).alias("customers_ratio"),
    )
    .with_columns(
        (pl.col("resstock_total_kwh_normalized") / pl.col("eia_residential_kwh")).alias("kwh_ratio"),
        (
            (pl.col("resstock_total_kwh_normalized") - pl.col("eia_residential_kwh"))
            / pl.col("eia_residential_kwh")
            * 100
        ).alias("kwh_pct_diff"),
    )
    .select(
        "utility_code",
        "buildings",
        "resstock_total_kwh",
        "customers_ratio",
        "resstock_total_kwh_normalized",
        "eia_residential_kwh",
        "kwh_ratio",
        "kwh_pct_diff",
    )
    .sort("resstock_total_kwh", descending=True)
)

print(f"ResStock vs EIA-861 residential kWh by utility (customer-count normalized; {STATE_UPPER}, {EIA_YEAR}, {RELEASE_SOURCE_LABEL}):")
display(kwh_comparison)

ResStock vs EIA-861 residential kWh by utility (customer-count normalized; MD, 2018, _sb (post-processed)):


utility_code,buildings,resstock_total_kwh,customers_ratio,resstock_total_kwh_normalized,eia_residential_kwh,kwh_ratio,kwh_pct_diff
str,u32,f64,f64,f64,f32,f64,f64
"""bge""",4897,1.5982e10,1.060855,1.5065e10,1.2948e10,1.163445,16.344547
"""pepco""",2344,7.3058e9,1.125224,6.4928e9,5.8435e9,1.111106,11.110635
"""poted""",1076,4.0710e9,1.149974,3.5400e9,3.3581e9,1.054179,5.417928
"""smeco""",630,2.7047e9,1.058488,2.5553e9,2.2768e9,1.122324,12.232354
"""dpl""",546,1.9062e9,0.771932,2.4694e9,2.2193e9,1.112699,11.269928
"""choptank""",475,1.5681e9,2.467383,6.3552e8,7.18323968e8,0.884728,-11.527203
"""hagerstown_muni""",25,1.1014e8,0.424379,2.5954e8,1.66743008e8,1.556505,55.650453
"""easton_muni""",1,3.0390e6,0.030198,1.0064e8,1.14926e8,0.875658,-12.434204


## Section 4: Assumptions and limitations

*Coming next — document weighting, residential-only EIA scope, year alignment, and
known discrepancy drivers.*